# 📘 MODULE 4 · Assignment M04.4: Data Quality Check (10 pts)
**Due in Module 4.** When you finish, upload your PDF to the Canvas assignment named **M04.4**.

**This week you:** find one real data-quality issue, fix it by choosing from a menu (no code editing), and defend your choice.

**Same 5 steps as every week:** 1 run setup → 2 fill in your project card → 3 load data → 4 do this week's TODO cells → 5 run the submission block and export the PDF.

**You're done when:** the detective cell shows a problem, your fix menu passes its self-check, the AFTER cell proves the fix, and the 📤 submission block prints cleanly.

🧰 **First time here?** You will not write any Python. Press ▶ on a cell to run it, and type words between the quotation marks in 🟨 TODO cells. 🟥 Red error text can't hurt anything. 🟨 Yellow "NOT DONE YET" is coaching, not an error. `Runtime → Run all` fixes most weird problems. Stuck for 15 minutes? Screenshot it and message me.

**Before anything else:** `File → Save a copy in Drive`, then rename it `HIM4331_M04.4_LastnameFirstname`.

## ▶ Step 1: setup (nothing to type, just press ▶)

In [ ]:
#@title ▶ Step 1: tools and self-check helpers (just run it)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import datetime

pd.set_option("display.max_columns", 60)

_CHECKS = {}   # every self-check records its latest result here; the submission block reports them

def check(condition, ok_msg, fix_msg):
    """Friendly self-check: green check if done, yellow hint if not."""
    _CHECKS[fix_msg] = (bool(condition), ok_msg)
    if condition:
        print("✅ " + ok_msg)
        return True
    else:
        print("🟨 NOT DONE YET → " + fix_msg)
        return False

def banner(module, step, points):
    print("=" * 62)
    print(f"📤 MODULE {module} | {step} SUBMISSION BLOCK  ({points} pts)")
    print("=" * 62)
    print(f"Student:              {STUDENT_NAME}")
    print(f"Dataset:              {DATASET_CHOICE}")
    print(f"My research question: {MY_RESEARCH_QUESTION}")
    print(f"My stakeholder:       {MY_STAKEHOLDER}")
    print(f"Generated:            {datetime.datetime.now():%Y-%m-%d %H:%M}")
    print("-" * 62)
    _n = len(_CHECKS)
    _fail = [(fix, ok) for fix, (passed, ok) in _CHECKS.items() if not passed]
    if _n and not _fail:
        print(f"SELF-CHECK STATUS: ✅ ALL {_n} CHECKS PASSED")
    else:
        print(f"SELF-CHECK STATUS: 🟨 {len(_fail)} OF {_n} NOT DONE")
    for fix, (passed, ok) in _CHECKS.items():
        print(("   ✅ " + ok) if passed else ("   🟨 " + fix))
    print("-" * 62)

def next_week_card():
    """Prints the exact lines to paste into next week's project card."""
    g = globals()
    print("-" * 62)
    print("📋 YOUR PROJECT CARD FOR NEXT WEEK (copy these lines into Step 2 next time):")
    print(f'    STUDENT_NAME         = "{STUDENT_NAME}"')
    print(f'    DATASET_CHOICE       = "{DATASET_CHOICE}"')
    print(f'    MY_RESEARCH_QUESTION = "{MY_RESEARCH_QUESTION}"')
    print(f'    MY_STAKEHOLDER       = "{MY_STAKEHOLDER}"')
    if g.get("M04_COLUMN") and g.get("M04_FIX"):
        print(f'    M04_COLUMN = "{g["M04_COLUMN"]}"')
        print(f'    M04_FIX    = "{g["M04_FIX"]}"')
        if g.get("FLAG_FROM_COLUMN"):
            print(f'    FLAG_FROM_COLUMN = "{g["FLAG_FROM_COLUMN"]}"')
            print(f'    FLAG_ABOVE       = "{g.get("FLAG_ABOVE", "")}"')
            print(f'    FLAG_YES_LABEL   = "{g.get("FLAG_YES_LABEL", "")}"')
            print(f'    FLAG_NO_LABEL    = "{g.get("FLAG_NO_LABEL", "")}"')

# Where the four datasets live (your instructor hosts these; do not edit)
DATA_URLS = {
    "diabetes":     "https://raw.githubusercontent.com/ctg30/him4331-data/main/diabetic_data.csv",
    "hospitals":    "https://raw.githubusercontent.com/ctg30/him4331-data/main/cms_hospitals.csv",
    "readmissions": "https://raw.githubusercontent.com/ctg30/him4331-data/main/cms_readmissions.csv",
    "breaches":     "https://raw.githubusercontent.com/ctg30/him4331-data/main/hhs_breaches.csv",
}
_MISS_CODES = ["?", "Unknown", "UNKNOWN", "Not Available", "N/A", "NA", "None", ""]
print("✅ Step 1 done. Tools loaded. Go to Step 2.")

## 🟨 Step 2: your project card
Open last week's PDF, find the box titled 📋 YOUR PROJECT CARD FOR NEXT WEEK, and paste those lines over the blank ones below. Then press ▶. Your question is allowed to get sharper as you learn; if it changed, use the newer version.

In [ ]:
#@title 🟨 Step 2: fill in your project card
try:
    STUDENT_NAME         = ""   # e.g. "Jordan Rivera"
    DATASET_CHOICE       = ""   # one of: "diabetes", "hospitals", "readmissions", "breaches"
    MY_RESEARCH_QUESTION = ""   # e.g. "Do longer hospital stays predict 30-day readmission for diabetic patients?"
    MY_STAKEHOLDER       = ""   # who you're writing for, e.g. "Quality Director at a 300-bed hospital"

    # ---- self-checks (don't edit below) ----
    DATASET_CHOICE = DATASET_CHOICE.strip().lower()
    check(len(STUDENT_NAME.strip()) > 2, "Name entered.", "Type your name in STUDENT_NAME.")
    check(DATASET_CHOICE in DATA_URLS, f"Dataset '{DATASET_CHOICE}' is valid.",
          "DATASET_CHOICE must be exactly one of: diabetes, hospitals, readmissions, breaches")
    check(len(MY_RESEARCH_QUESTION.strip()) > 15 and MY_RESEARCH_QUESTION.strip().endswith("?"),
          "Research question entered.",
          "Write your research question as one full sentence ending in '?'")
    check(len(MY_STAKEHOLDER.strip()) > 5, "Stakeholder entered.",
          "Name a real decision-maker role, e.g. 'HIM privacy officer'.")

except (KeyError, NameError, AttributeError, TypeError, IndexError):
    print("🟨 This cell needs an earlier cell finished first.")
    print("   Scroll up, complete the 🟨 TODO cells above in order, then re-run this cell.")
    print("   (Fastest fix: Runtime → Run all)")

## ▶ Step 3: load your data (nothing to type, just press ▶)

In [ ]:
#@title ▶ Step 3: load your dataset (just run it)
try:
    try:
        df = pd.read_csv(DATA_URLS[DATASET_CHOICE], low_memory=False)
        df_clean = df.copy()
        print(f"✅ Step 3 done. Loaded '{DATASET_CHOICE}': {df.shape[0]:,} rows × {df.shape[1]} columns")
    except KeyError:
        print("🟨 Fix DATASET_CHOICE in Step 2 first, then re-run this cell.")
    except Exception as e:
        print("🟨 Could not load the data. Check your internet connection and re-run.")
        print("   If it still fails, screenshot this and message your instructor:", repr(e))

except (KeyError, NameError, AttributeError, TypeError, IndexError):
    print("🟨 This cell needs an earlier cell finished first.")
    print("   Scroll up, complete the 🟨 TODO cells above in order, then re-run this cell.")
    print("   (Fastest fix: Runtime → Run all)")

## 🟨 Step 4: this week's work
💡 **Known trouble spots in YOUR file.** I've checked all four, so these are real:

* **diabetes** → four columns hide missing values as `"?"`: `weight` (97% missing!), `medical_specialty` (~49%), `payer_code` (~40%), `race` (~2%). Also `max_glu_serum` and `A1Cresult` are genuinely blank for most patients. → try fix `recode_missing`, or `drop_rows` if it's a column you must have.
* **hospitals** → `Hospital overall rating` is stored as TEXT because ~1,000 hospitals say `Not Available` instead of a 1 to 5 star number. You cannot do math on it until it's converted. → fix `to_number`. (You will need this fix in M06 and M07 if the rating is part of your question.)
* **readmissions** → `Number of Discharges` is blank on 10,088 of 18,330 rows, and `Excess Readmission Ratio` is blank on 6,610 (CMS suppresses small-volume hospitals). → fix `drop_rows` on the column your question needs.
* **breaches** → there are exactly 2 duplicate rows (→ fix `drop_duplicates`), and `Individuals Affected` has wild outliers: the median breach affects ~6,500 people but the largest affects 192.7 million, which will distort every average you compute. → `document_only` is a defensible choice here if you explain the outlier.

📝 **Write down the column and fix you choose.** You will type them into your project card every week from Module 5 on, so the later notebooks work on the same cleaned data.

In [ ]:
#@title ▶ STEP A: The Detective Cell finds candidate problems for you (just run it)
try:
    print("🔍 A) MISSING VALUES per column (top 15):")
    print(df.isna().sum().sort_values(ascending=False).head(15).to_string())
    print()
    print("🔍 B) DUPLICATE ROWS:", df.duplicated().sum())
    print()
    print("🔍 C) COLUMNS THAT MIGHT HIDE MISSING VALUES AS CODES like '?', 'Unknown', 'Not Available', 'N/A':")
    suspects = []
    for col in df.columns:
        if not pd.api.types.is_numeric_dtype(df[col]):
            vals = df[col].astype(str).str.strip()
            hits = vals.isin(_MISS_CODES).sum()
            if hits > 0:
                suspects.append((col, int(hits)))
    for col, hits in sorted(suspects, key=lambda x: -x[1])[:10]:
        print(f"   {col}: {hits:,} suspicious values")
    if not suspects:
        print("   (none found; focus on A or B above)")

except (KeyError, NameError, AttributeError, TypeError, IndexError):
    print("🟨 This cell needs an earlier cell finished first.")
    print("   Scroll up, complete the 🟨 TODO cells above in order, then re-run this cell.")
    print("   (Fastest fix: Runtime → Run all)")

In [ ]:
#@title 🟨 TODO STEP B: name the column and pick your fix from the menu
try:
    M04_COLUMN = ""   # the column with the issue, copied exactly from the detective output
    M04_FIX    = ""   # pick ONE (type it exactly):
                      #   "recode_missing"  → turn codes like '?' / 'Not Available' into proper missing values
                      #   "drop_rows"       → remove rows that are missing this column
                      #   "impute_median"   → fill missing numbers with the column's median
                      #   "to_number"       → convert a text column to numbers (bad values become missing)
                      #   "drop_duplicates" → remove duplicate rows (M04_COLUMN still required, for reporting)
                      #   "document_only"   → keep the data as-is and explain why that's the right call

    _FIX_MENU = ["recode_missing", "drop_rows", "impute_median", "to_number",
                 "drop_duplicates", "document_only"]
    M04_COLUMN = M04_COLUMN.strip()
    M04_FIX = M04_FIX.strip().lower()
    check(M04_COLUMN in df.columns, "Column found.",
          "M04_COLUMN must be an exact column name from the detective output above.")
    check(M04_FIX in _FIX_MENU, f"Fix choice '{M04_FIX}' is valid.",
          "M04_FIX must be exactly one of: " + ", ".join(_FIX_MENU))

except (KeyError, NameError, AttributeError, TypeError, IndexError):
    print("🟨 This cell needs an earlier cell finished first.")
    print("   Scroll up, complete the 🟨 TODO cells above in order, then re-run this cell.")
    print("   (Fastest fix: Runtime → Run all)")

In [ ]:
#@title ▶ STEP C: BEFORE → apply your fix → AFTER (just run it; the notebook does the coding)
try:
    def _diagnose(frame, col):
        s = frame[col]
        coded = s.astype(str).str.strip().isin(_MISS_CODES).sum() if not pd.api.types.is_numeric_dtype(s) else 0
        return (f"missing={s.isna().sum():,} | coded-as-missing={coded:,} | "
                f"duplicated rows in table={frame.duplicated().sum():,} | dtype={s.dtype}")

    print("BEFORE:", _diagnose(df, M04_COLUMN))

    df_clean = df.copy()
    if M04_FIX == "recode_missing":
        df_clean[M04_COLUMN] = df_clean[M04_COLUMN].replace(_MISS_CODES, np.nan)
    elif M04_FIX == "drop_rows":
        df_clean[M04_COLUMN] = df_clean[M04_COLUMN].replace(_MISS_CODES, np.nan)
        df_clean = df_clean.dropna(subset=[M04_COLUMN])
    elif M04_FIX == "impute_median":
        df_clean[M04_COLUMN] = pd.to_numeric(
            df_clean[M04_COLUMN].replace(_MISS_CODES, np.nan), errors="coerce")
        df_clean[M04_COLUMN] = df_clean[M04_COLUMN].fillna(df_clean[M04_COLUMN].median())
    elif M04_FIX == "to_number":
        df_clean[M04_COLUMN] = pd.to_numeric(
            df_clean[M04_COLUMN].replace(_MISS_CODES, np.nan), errors="coerce")
    elif M04_FIX == "drop_duplicates":
        df_clean = df_clean.drop_duplicates()
    elif M04_FIX == "document_only":
        pass  # keeping data as-is, on purpose

    print("AFTER: ", _diagnose(df_clean, M04_COLUMN))
    print(f"Rows: {len(df):,} → {len(df_clean):,}")

except (KeyError, NameError, AttributeError, TypeError, IndexError):
    print("🟨 This cell needs an earlier cell finished first.")
    print("   Scroll up, complete the 🟨 TODO cells above in order, then re-run this cell.")
    print("   (Fastest fix: Runtime → Run all)")

### ✍️ What a good explanation looks like
Flu-clinic example:

> "The insurance column recorded 2,100 patients as 'Unknown' instead of leaving the field blank, which would have silently inflated the 'insured' counts in my analysis. I recoded those values as properly missing rather than dropping the rows, because those patients' other fields (age, doses) are still valid and useful. Dropping them would have thrown away 17% of my data to fix one column."

Pattern: **what the issue was and why it's dangerous → what I did → why my choice beats the alternative.**

In [ ]:
#@title 🟨 TODO STEP D: defend your call (finish the starters)
M04_EXPLANATION = """
The issue I found was ...
If left alone, it would have distorted my analysis by ...
I chose to [my fix] instead of the alternatives because ...
"""

check("I found was ..." not in M04_EXPLANATION
      and "analysis by ..." not in M04_EXPLANATION
      and "[my fix]" not in M04_EXPLANATION,
      "Explanation written.", "Finish all three sentence starters with your own words (replace [my fix] too).")

### 🔧 OPTIONAL STEP E: turn a number into a yes/no group (MY_FLAG)
*Skip this unless you need it.* Some questions need a category where your data only has a number. Example: "which hospitals perform WORSE than expected?" is a yes/no group built from the number `Excess Readmission Ratio` (above 1.0 = worse).

**You'll want this if:** your dataset is **readmissions** (needed, otherwise Module 5 has no meaningful category pair), or your dataset is **hospitals** or **breaches** and you want a yes/no outcome, or you plan to run a logistic regression in Module 7.

**Skip it if your dataset is diabetes.** You already have `readmitted`, `change`, `diabetesMed`, and `insulin` as ready-made categories.

Ready-made settings you can copy:

* **readmissions** → column `Excess Readmission Ratio`, cutoff `1.0`, yes-label `Worse than expected`, no-label `As expected or better`
* **breaches** → column `Individuals Affected`, cutoff `10000`, yes-label `Large breach`, no-label `Smaller breach`

📝 If you use it, write these four settings down too. You will re-enter them in your project card from Module 5 on.

In [ ]:
#@title 🟨 OPTIONAL STEP E: make a yes/no column called MY_FLAG (leave blank to skip)
try:
    FLAG_FROM_COLUMN = ""   # a NUMBER column, e.g. "Excess Readmission Ratio"
    FLAG_ABOVE       = ""   # the cutoff, in quotes, e.g. "1.0"
    FLAG_YES_LABEL   = ""   # label for rows ABOVE the cutoff, e.g. "Worse than expected"
    FLAG_NO_LABEL    = ""   # label for rows at or below,     e.g. "As expected or better"

    FLAG_FROM_COLUMN = FLAG_FROM_COLUMN.strip()
    FLAG_ABOVE = FLAG_ABOVE.strip()
    if FLAG_FROM_COLUMN == "" and FLAG_ABOVE == "":
        print("⏭️  Skipped. You don't need MY_FLAG.")
    elif FLAG_FROM_COLUMN not in df_clean.columns:
        print(f"🟨 '{FLAG_FROM_COLUMN}' isn't a column in your dataset. Copy the exact name, or clear all four boxes to skip.")
    else:
        _num = pd.to_numeric(df_clean[FLAG_FROM_COLUMN], errors="coerce")
        try:
            _cut = float(FLAG_ABOVE)
        except ValueError:
            _cut = None
            print(f"🟨 FLAG_ABOVE ('{FLAG_ABOVE}') needs to be a number in quotes, like \"1.0\".")
        if _cut is not None:
            _yes = FLAG_YES_LABEL.strip() or "Above cutoff"
            _no = FLAG_NO_LABEL.strip() or "At or below cutoff"
            _flag = pd.Series([None] * len(df_clean), index=df_clean.index, dtype=object)
            _flag[_num.notna() & (_num > _cut)] = _yes
            _flag[_num.notna() & (_num <= _cut)] = _no
            df_clean["MY_FLAG"] = _flag
            print(f"✅ Created a new column called MY_FLAG from '{FLAG_FROM_COLUMN}' (cutoff {_cut}). Counts:")
            print(df_clean["MY_FLAG"].value_counts(dropna=False).to_string())
            print("   (Rows shown as None/NaN had no number to compare and are left out of later tests.)")

except (KeyError, NameError, AttributeError, TypeError, IndexError):
    print("🟨 This cell needs an earlier cell finished first.")
    print("   Scroll up, complete the 🟨 TODO cells above in order, then re-run this cell.")
    print("   (Fastest fix: Runtime → Run all)")

## 📤 Step 5: submission block

In [ ]:
#@title 📤 M04.4 SUBMISSION BLOCK (Module 4): run, then File → Print → Save as PDF
try:
    banner(4, "M04.4 Data Quality Check", 10)
    print("1) THE ISSUE COLUMN:", M04_COLUMN)
    print("2) MY FIX:", M04_FIX)
    print("3) BEFORE:", _diagnose(df, M04_COLUMN))
    print("   AFTER: ", _diagnose(df_clean, M04_COLUMN))
    print(f"   Rows: {len(df):,} → {len(df_clean):,}")
    print()
    print("4) MY EXPLANATION:")
    print(M04_EXPLANATION.strip())
    print()
    print("5) MY_FLAG:", f"built from '{FLAG_FROM_COLUMN}' (cutoff {FLAG_ABOVE})" if FLAG_FROM_COLUMN else "not used")
    next_week_card()
    print("=" * 62)
    print("📎 Now: File → Print → Save as PDF → upload to the M04.4 Canvas assignment (Module 4).")
    print("📝 Keep this PDF handy: from Module 5 on you will paste the project card lines above into Step 2.")

except (KeyError, NameError, AttributeError, TypeError, IndexError):
    print("🟨 This cell needs an earlier cell finished first.")
    print("   Scroll up, complete the 🟨 TODO cells above in order, then re-run this cell.")
    print("   (Fastest fix: Runtime → Run all)")

## 📎 Step 5: export and submit
1. Make sure the submission block above printed with **no 🟨 warnings**.
2. `File → Print` (or Ctrl+P / Cmd+P) → choose **Save as PDF** → save it as `HIM4331_M04.4_LastnameFirstname.pdf`.
3. Upload that PDF to the Canvas assignment named **M04.4** (Module 4).
4. Keep this notebook. You will copy your project card from it next week.